In [14]:
# 1. Load and prepare the data


import pandas as pd
import numpy as np


df = pd.read_csv("../0_data/1_cleaned/fact_inventory.csv")

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,avg_shipping_delay,late_deliveries
0,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,1.19,0.39,0.331,64,1.02,3.89,0.95,38
1,24,Elevation Training Mask 2.0,Soccer,Fitness,4.13,2.53,0.613,231,3.12,3.55,0.51,37
2,35,adidas Brazuca 2014 Official Match Ball,Baseball & Softball,Fitness,1.25,0.48,0.384,65,1.00,3.35,0.46,36
3,37,adidas Kids' F5 Messi FG Soccer Cleat,Baseball & Softball,Fitness,3.37,1.88,0.558,781,2.98,3.46,0.67,155
4,44,adidas Men's F10 Messi TRX FG Soccer Cleat,Baseball & Softball,Fitness,3.60,2.22,0.617,939,3.10,3.51,0.54,158


In [15]:
# 2. Classify products into ABC categories


# Sort products by demand.
df = df.sort_values(
    "total_demand",
    ascending=False
).reset_index(drop=True)


# Demand percentage
df["demand_share_pct"] = (
    df["total_demand"]
    / df["total_demand"].sum()
) * 100


# Cumulative demand percentage
df["cumulative_share_pct"] = (
    df["demand_share_pct"]
    .cumsum()
)


# Classify products into ABC categories based on cumulative demand percentage
def classify_abc(x):

    if x <= 80:
        return "A"

    elif x <= 95:
        return "B"

    else:
        return "C"

df["ABC_Class"] = (
    df["cumulative_share_pct"]
    .apply(classify_abc)
)

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,avg_shipping_delay,late_deliveries,demand_share_pct,cumulative_share_pct,ABC_Class
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,0.58,13473,19.188240,19.188240,A
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,0.56,11476,16.391419,35.579659,A
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,0.56,10565,15.049768,50.629428,A
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,0.55,6629,9.550119,60.179546,A
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,0.58,5865,8.262623,68.442169,A


In [16]:
# 3. Classify products into XYZ categories


# def classify_xyz(cv):

#     if cv <= 0.5:
#         return "X"

#     elif cv <= 1:
#         return "Y"

#     else:
#         return "Z"

# df["XYZ_Class"] = (df["coefficient_of_variation"].apply(classify_xyz))

z_threshold = df["coefficient_of_variation"].quantile(0.90)

def classify_xyz(cv):

    if cv >= z_threshold:
        return "Z"
    
    elif cv >= 0.5:
        return "Y"
    
    else:
        return "X"
    
df["XYZ_Class"] = (df["coefficient_of_variation"].apply(classify_xyz))

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,avg_shipping_delay,late_deliveries,demand_share_pct,cumulative_share_pct,ABC_Class,XYZ_Class
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,0.58,13473,19.188240,19.188240,A,X
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,0.56,11476,16.391419,35.579659,A,X
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,0.56,10565,15.049768,50.629428,A,X
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,0.55,6629,9.550119,60.179546,A,X
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,0.58,5865,8.262623,68.442169,A,X


In [17]:
# 4. Combine ABC and XYZ classifications into a single column


df["ABC_XYZ"] = (
    df["ABC_Class"]
    + df["XYZ_Class"]
)

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,avg_shipping_delay,late_deliveries,demand_share_pct,cumulative_share_pct,ABC_Class,XYZ_Class,ABC_XYZ
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,0.58,13473,19.188240,19.188240,A,X,AX
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,0.56,11476,16.391419,35.579659,A,X,AX
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,0.56,10565,15.049768,50.629428,A,X,AX
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,0.55,6629,9.550119,60.179546,A,X,AX
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,0.58,5865,8.262623,68.442169,A,X,AX


In [18]:
# 5. Forecast demand for each product and merge into the inventory table


# Merge the average forecast from fact_forecast.csv
forecast = pd.read_csv("../3_outputs/fact_forecast.csv")


# Calculate the average forecast over the next 30 days for each product
forecast_summary = (
    forecast.groupby("product_id")["forecast_demand"]
            .mean()
            .reset_index()
            .rename(columns={
                "forecast_demand": "forecast_daily_demand"
            })
)


# Merge into the inventory table
df = df.merge(
    forecast_summary,
    on="product_id",
    how="left"
)

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,avg_shipping_delay,late_deliveries,demand_share_pct,cumulative_share_pct,ABC_Class,XYZ_Class,ABC_XYZ,forecast_daily_demand
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,0.58,13473,19.188240,19.188240,A,X,AX,72.233333
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,0.56,11476,16.391419,35.579659,A,X,AX,62.633333
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,0.56,10565,15.049768,50.629428,A,X,AX,56.100000
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,0.55,6629,9.550119,60.179546,A,X,AX,36.133333
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,0.58,5865,8.262623,68.442169,A,X,AX,31.133333


In [19]:
# 6. Lead time


df["lead_time_days"] = (
    df["avg_shipping_days"]
      .round()
      .astype(int)
)

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,avg_shipping_delay,late_deliveries,demand_share_pct,cumulative_share_pct,ABC_Class,XYZ_Class,ABC_XYZ,forecast_daily_demand,lead_time_days
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,0.58,13473,19.188240,19.188240,A,X,AX,72.233333,4
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,0.56,11476,16.391419,35.579659,A,X,AX,62.633333,3
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,0.56,10565,15.049768,50.629428,A,X,AX,56.100000,4
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,0.55,6629,9.550119,60.179546,A,X,AX,36.133333,3
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,0.58,5865,8.262623,68.442169,A,X,AX,31.133333,3


In [20]:
# 7. Safety stock calculation


# Use a 95% service level (Z = 1.65)
# Safety Stock = Z × σ × √Lead Time
SERVICE_LEVEL = 1.65

df["safety_stock"] = (
    SERVICE_LEVEL
    * df["demand_stddev"]
    * np.sqrt(df["lead_time_days"])
).round().astype(int)

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,avg_shipping_delay,late_deliveries,demand_share_pct,cumulative_share_pct,ABC_Class,XYZ_Class,ABC_XYZ,forecast_daily_demand,lead_time_days,safety_stock
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,0.58,13473,19.188240,19.188240,A,X,AX,72.233333,4,55
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,0.56,11476,16.391419,35.579659,A,X,AX,62.633333,3,46
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,0.56,10565,15.049768,50.629428,A,X,AX,56.100000,4,49
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,0.55,6629,9.550119,60.179546,A,X,AX,36.133333,3,35
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,0.58,5865,8.262623,68.442169,A,X,AX,31.133333,3,31


In [21]:
# 8. Lead time demand (based on the forecast, not historical demand)


df["forecast_daily_demand"] = df["forecast_daily_demand"].fillna(0)

df["lead_time_demand"] = (
    df["forecast_daily_demand"]
    * df["lead_time_days"]
).round().astype(int)

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,...,late_deliveries,demand_share_pct,cumulative_share_pct,ABC_Class,XYZ_Class,ABC_XYZ,forecast_daily_demand,lead_time_days,safety_stock,lead_time_demand
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,...,13473,19.188240,19.188240,A,X,AX,72.233333,4,55,289
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,...,11476,16.391419,35.579659,A,X,AX,62.633333,3,46,188
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,...,10565,15.049768,50.629428,A,X,AX,56.100000,4,49,224
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,...,6629,9.550119,60.179546,A,X,AX,36.133333,3,35,108
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,...,5865,8.262623,68.442169,A,X,AX,31.133333,3,31,93


In [22]:
# 9. Reorder point


# Reorder Point = Lead Time Demand + Safety Stock
df["reorder_point"] = (
    df["lead_time_demand"]
    + df["safety_stock"]
)

df.head()

,product_id,product_name,category_name,department_name,avg_daily_demand,demand_stddev,coefficient_of_variation,total_demand,units_per_day,avg_shipping_days,...,demand_share_pct,cumulative_share_pct,ABC_Class,XYZ_Class,ABC_XYZ,forecast_daily_demand,lead_time_days,safety_stock,lead_time_demand,reorder_point
0,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,73.26,16.72,0.228,73698,3.62,3.51,...,19.188240,19.188240,A,X,AX,72.233333,4,55,289,344
1,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,62.58,15.98,0.255,62956,3.52,3.48,...,16.391419,35.579659,A,X,AX,62.633333,3,46,188,234
2,1014,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,Fan Shop,57.46,14.81,0.258,57803,3.48,3.50,...,15.049768,50.629428,A,X,AX,56.100000,4,49,224,273
3,191,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,Footwear,36.46,12.09,0.332,36680,3.31,3.49,...,9.550119,60.179546,A,X,AX,36.133333,3,35,108,143
4,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,31.55,10.68,0.339,31735,3.23,3.49,...,8.262623,68.442169,A,X,AX,31.133333,3,31,93,124


In [23]:
# 10. Create the final fact_inventory_analysis table and save to CSV


fact_inventory_analysis = df[
    [
        "product_id",
        "product_name",
        "category_name",
        "department_name",
        "total_demand",
        "forecast_daily_demand",
        "ABC_Class",
        "XYZ_Class",
        "ABC_XYZ",
        "lead_time_days",
        "demand_stddev",
        "safety_stock",
        "lead_time_demand",
        "reorder_point"
    ]
]

fact_inventory_analysis.to_csv(
    "../3_outputs/fact_inventory_analysis.csv",
    index=False
)